# Bayesian Online Changepoint Detection

## Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as st

## Inputs

In [3]:
def constant_hazard(lam, r):
    return 1/lam * np.ones(r.shape)

## Function

In [2]:
def bocd(data, observation_likelihood, hazard_function, pts_interval):
    n_data = len(data)
    maxes = np.zeros(n_data + 1)
    # Initialize lower triangular matrix representing the posterior as
    # function of time. Model parameters are initialized in the model class.
    R = np.zeros((n_data + 1, n_data + 1))
    R[0, 0] = 1
    for t, x in enumerate(data):
        # Evaluate the predictive distribution for the new datum under each of
        # the parameters.
        pred_probs = observation_likelihood.pdf(x)
        # Evaluate the hazard function for this interval
        haz_f = hazard_function(np.array(range(t + 1)))
        # Evaluate the growth probabilities - shift the probabilities down and to
        # the right, scaled by the hazard function and the predictive
        # probabilities.
        R[1:t+2, t+1] = R[0:t+1, t] * pred_probs * (1 - haz_f)
        # Evaluate the probability that there *was* a changepoint and we're
        # accumulating the mass back down at r = 0.
        R[0, t+1] = np.sum( R[0:t+1, t] * pred_probs * haz_f)
        # Renormalize the run length probabilities for improved numerical
        # stability.
        R[:, t+1] = R[:, t+1] / np.sum(R[:, t+1])
        # Update the parameter sets for each possible run length.
        observation_likelihood.update_theta(x)
        # Calculate max for t
        maxes[t] = R[:, t].argmax()
    return R, maxes

## Models / Likelihoods

In [4]:
class StudentT:
    def __init__(self, alpha, beta, kappa, mu):
        self.alpha0 = self.alpha = np.array([alpha])
        self.beta0 = self.beta = np.array([beta])
        self.kappa0 = self.kappa = np.array([kappa])
        self.mu0 = self.mu = np.array([mu])

    def pdf(self, data):
        return st.t.pdf(x=data, 
                           df=2*self.alpha,
                           loc=self.mu,
                           scale=np.sqrt(self.beta * (self.kappa+1) / (self.alpha *
                               self.kappa)))

    def update_theta(self, data):
        muT0 = np.concatenate((self.mu0, (self.kappa * self.mu + data) / (self.kappa + 1)))
        kappaT0 = np.concatenate((self.kappa0, self.kappa + 1.))
        alphaT0 = np.concatenate((self.alpha0, self.alpha + 0.5))
        betaT0 = np.concatenate((self.beta0, self.beta + (self.kappa * (data -
            self.mu)**2) / (2. * (self.kappa + 1.))))
            
        self.mu = muT0
        self.kappa = kappaT0
        self.alpha = alphaT0
        self.beta = betaT0

In [5]:
class Normal:
    def __init__(self, mu, prec):
        self.mu0 = self.mu = np.array([mu])
        self.prec0 = self.prec = np.array([prec])

    def pdf(self, data):
        return st.norm.pdf(x=data, loc=self.loc, scale=1 / self.scale)

    def update_theta(self, data):
        offsets = np.arange(1, len(self.mu) + 1)
        self.mu = np.concatenate((self.mu0, (self.mu * offsets + data) / (offsets + 1)))
        self.prec = np.concatenate((self.prec0, self.prec + 1.))

## Tests

In [6]:
data = np.concatenate((np.full(200, 10), np.full(300, 20)))

In [7]:
model = Normal(0, 0.2)